# MCB Annotation Notebook
## Machine-Assisted Reading of Jack Kerouac's *Mexico City Blues*
### LLM-Supported Theme and Tone Annotation — MAQD Framework

---

**Model:** Mistral 7B via Ollama  
**Corpus:** 244 choruses (including 216-A, 216-B, 216-C)  
**Task:** Closed-set classification — 5 tone labels, 6 theme labels  
**Output:** JSON per chorus → stored in annotations CSV

Pipeline:
1. Load and inspect the chorus CSV
2. Sort choruses into correct sequence
3. Flag structurally unique choruses
4. Define the system prompt (coding scheme)
5. Define helper functions (following Week 10 pattern)
6. Single-chorus connectivity test
7. Run annotation loop across all 244 choruses

In [ ]:
import ollama
import pandas as pd
import json
import re
from pathlib import Path
from tqdm.notebook import tqdm

print("Libraries loaded.")

In [ ]:
# ── Model configuration ───────────────────────────────────────────────
# Mistral 7B via Ollama — same model used in Week 10 Demo 2 and Demo 3.
# Ensure Ollama is running (ollama serve) before executing this cell.
# ─────────────────────────────────────────────────────────────────────

TEXT_MODEL  = "mistral"

# Temperature = 0.0 for fully deterministic annotation output.
# Unlike Week 10's rewriting tasks (temperature 0.3–0.4),
# annotation requires the same chorus to always receive the same label.
TEMPERATURE = 0.0

In [ ]:
# ── Connectivity check ────────────────────────────────────────────────
# Verify Mistral is available before loading data.
# Following Week 10's pattern of checking the model before any work.
# ─────────────────────────────────────────────────────────────────────

test = ollama.chat(
    model=TEXT_MODEL,
    messages=[{
        "role": "user",
        "content": 'Reply with only this JSON and nothing else: {"tone": "T1", "theme": "TH1"}'
    }],
    options={"temperature": 0.0},
)

raw = test["message"]["content"].strip()
print("Raw model response:", repr(raw))
print()

# Note whether the model wraps output in backticks —
# the parse function handles this, but it is useful to know
if raw.startswith("```"):
    print("NOTE: model is wrapping output in markdown backticks.")
    print("      parse_annotation() will strip these automatically.")
else:
    print("✓ Clean JSON output — no backtick wrapping.")

---
## Load and Inspect the Corpus

The CSV contains 244 choruses including the three-part 216th Chorus
(216-A, 216-B, 216-C), consistent with Jones (1992, Ch. 8, p. 170).

**Known properties of this CSV (from pre-analysis):**
- Two columns: `chorus_id` (string dtype) and `text`
- No null values, no empty rows, no duplicate IDs
- Text preserves Kerouac's lineation as `\n` characters
- Non-ASCII characters present: smart quotes, en dashes, accented
  letters — these are Kerouac's original punctuation, not errors,
  and are preserved throughout
- `chorus_id` is string dtype throughout because 216-A/B/C cannot
  be integers — all ID comparisons must use `.astype(str)`

In [ ]:
# ── Load corpus ───────────────────────────────────────────────────────
CORPUS_CSV = Path("../data/mexico_city_blues_choruses.csv")

assert CORPUS_CSV.exists(), (
    f"Cannot find: {CORPUS_CSV.resolve()}\n"
    f"Make sure the CSV is in the same folder as this notebook."
)

df = pd.read_csv(CORPUS_CSV, dtype={"chorus_id": str})

# Confirm basic shape
print(f"Loaded {len(df)} choruses.")
print(f"Columns: {list(df.columns)}")
print(f"chorus_id dtype: {df['chorus_id'].dtype}")
print()
print("First 5 rows:")
print(df[["chorus_id", "text"]].head())

In [ ]:
# ── Corpus overview ───────────────────────────────────────────────────
df["text_len"] = df["text"].str.len()

print("=== Mexico City Blues — Corpus Overview ===")
print(f"  Total choruses:      {len(df)}")
print(f"  Unique chorus IDs:   {df['chorus_id'].nunique()}")
print(f"  Shortest chorus:     {df['text_len'].min()} chars  "
      f"(Chorus {df.loc[df['text_len'].idxmin(), 'chorus_id']})")
print(f"  Longest chorus:      {df['text_len'].max()} chars  "
      f"(Chorus {df.loc[df['text_len'].idxmax(), 'chorus_id']})")
print(f"  Mean chorus length:  {df['text_len'].mean():.0f} chars")
print()

# Confirm the three-part 216th chorus is present
chorus_216 = df[df["chorus_id"].str.startswith("216")]
print(f"  Three-part 216th Chorus present: {len(chorus_216)} parts")
print(f"  IDs: {chorus_216['chorus_id'].tolist()}")

In [ ]:
# ── Sort choruses into correct reading sequence ───────────────────────
# String sort fails: '10' < '2' alphabetically.
# This key extracts the numeric part and the letter suffix separately.
# ─────────────────────────────────────────────────────────────────────

def sort_key(chorus_id):
    """
    Convert a chorus_id string into a sortable tuple.
    '216-A' → (216, 'A'), '87' → (87, '')
    """
    parts = str(chorus_id).split("-")
    return (int(parts[0]), parts[1] if len(parts) > 1 else "")

df["_sort_key"] = df["chorus_id"].apply(sort_key)
df = df.sort_values("_sort_key").reset_index(drop=True)
df = df.drop(columns=["_sort_key"])

print("Choruses sorted into correct sequence.")
print("First 5 IDs:", df["chorus_id"].head().tolist())
print("Last 5 IDs: ", df["chorus_id"].tail().tolist())
print()

# Verify the 216 sub-choruses are in order
idx_216 = df[df["chorus_id"].str.startswith("216")].index.tolist()
print(f"216th Chorus position in sequence: rows {idx_216}")
print(df.loc[idx_216, "chorus_id"].tolist())

In [ ]:
# ── Flag structurally unique choruses ────────────────────────────────
# Rest choruses (Jones, Ch. 7, pp. 158–159): three choruses defined
# by blank spaces and parenthetical stage directions.
# These are annotated T1/TH6 regardless of model output.
#
# NOTE: the 36th Chorus in this CSV contains the full preceding text
# (455 chars) before its rest-chorus marker. It is flagged here for
# interpretive purposes but is not anomalously short in this dataset.
# ─────────────────────────────────────────────────────────────────────

REST_CHORUSES = ["11", "36", "138"]
df["rest_chorus"] = df["chorus_id"].astype(str).isin(REST_CHORUSES)

print(f"Rest choruses flagged: {df['rest_chorus'].sum()} choruses")
print()

# Confirm each rest chorus is present and show its key marker
for rid in REST_CHORUSES:
    row = df[df["chorus_id"] == rid].iloc[0]
    # Find the parenthetical stage direction
    marker = re.search(r'\(.*?\)', row["text"])
    marker_text = marker.group(0) if marker else "no parenthetical found"
    print(f"  Chorus {rid} ({row['text_len']} chars) — marker: {marker_text}")

In [ ]:
# ── Prepare annotation columns ────────────────────────────────────────
# Added before the loop so the checkpoint CSV always has
# the same structure from the first save onwards.
# ─────────────────────────────────────────────────────────────────────

new_cols = {
    "model_tone":          None,   # LLM primary tone label (T1–T5)
    "model_theme":         None,   # LLM primary theme label (TH1–TH6)
    "model_raw":           None,   # raw response string (for inspection)
    "annotation_success":  None,   # True/False — did JSON parsing succeed?
    "human_tone":          None,   # your gold standard tone (50 choruses)
    "human_theme":         None,   # your gold standard theme (50 choruses)
    "ambiguity_flag":      None,   # 0=clear, 1=dual, 2=undecidable
}

for col, default in new_cols.items():
    if col not in df.columns:
        df[col] = default

print("Annotation columns added.")
print("Full column list:", list(df.columns))

---
## Prompt Design

The system prompt carries the full coding scheme as a fixed constant.
It is defined once and reused identically for all 244 choruses.

Three deliberate choices, following Week 10's prompt design principles:

1. **Fixed vocabulary** — the model receives a closed label set and
   is told to use only those codes, preventing invented categories.
2. **Format specification** — JSON output only, no explanation,
   no markdown, making responses programmatically parseable.
3. **Temperature 0.0** — unlike Week 10's rewriting demos (0.3–0.4),
   annotation requires the same chorus to always receive the same label.

Only the Definition and Surface Signal Markers from the coding scheme
are included — the dialectical framework is withheld deliberately.
The gap between the model's surface-level classification and the
human annotator's Jones-informed reading is the analytical object
of RQ2, not a design flaw.

In [ ]:
# ── System prompt — coding scheme ────────────────────────────────────
# Carries the five tone and six theme category definitions.
# Structured as system/user messages following Week 10's pattern.
# ─────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are a literary annotation assistant. You will receive \
one chorus from Jack Kerouac's Mexico City Blues at a time. Assign each chorus \
exactly one dominant tone label and one dominant theme label from the \
closed sets below.

TONE CATEGORIES — choose exactly one:
T1 CALM/MEDITATIVE: The chorus projects stillness or contemplative clarity. \
The singer-voice is unhurried. Buddhist vocabulary dominates: anatta, Tathagata, \
void, emptiness, nirvana, no-self, balloon, silence. Syntax is open and \
non-agitated. The 'I' is dissolved rather than asserted.

T2 ECSTATIC/IMPROVISATORY: The chorus enacts jazz-inflected immediacy. Language \
accelerates into sound-play. Dense oo-clusters, percussive coinages, \
onomatopoeia, scat riffs. First-person and kinetic. A music reference or \
musician name is present. Outward and fast-moving.

T3 ANXIOUS/RESTLESS: The chorus registers psychological pressure. The ego is \
asserting or defending itself. Oscillating syntax without resolution. \
Biographical proper nouns: Lowell, Memere, Leo, Joan, Jan. Guilt vocabulary. \
Lack of Buddhist terminology. Restless and kinetic.

T4 COMIC/ABSURDIST: The chorus deploys humour, wordplay, or theatrical \
absurdity. A named persona appears: Kerouaco, Bojangles Banghard, Asphasiax. \
Theatrical stage directions in parentheses. Mock-Spanish coinages. \
Self-deprecating gestures. Ironic distance from the subject.

T5 DESPAIRING/DARK: The chorus registers grief, mortality, or resignation. \
Mortality vocabulary: death, grave, dying, slaughter, sickness, blood. \
Dark imagery. Subdued pace. Down-and-out register. No Buddhist framing \
of the darkness.

THEME CATEGORIES — choose exactly one:
TH1 SPIRITUALITY/BUDDHISM & CATHOLICISM: Buddhist doctrine, anatta, Tathagata, \
bodhisattva, sutra, dharma, nirvana, compassion, no-self. Catholic vocabulary: \
Gerard-as-saint, Virgin Mary, confessional register. Light/dark as \
metaphysical argument.

TH2 JAZZ/MUSIC/BOP POETICS: Named jazz musicians (Parker, Lester Young, \
Sarah Vaughan). Words: bop, blues, jazz, horn, blow, chorus, riff. \
Percussive onomatopoeia. Direct music-to-writing analogy. A musician is \
named or bop phrase structure is enacted.

TH3 AUTOBIOGRAPHY/MEMORY/FAMILY: Biographical events. Lowell. Family members \
as narrative characters: Memere, Leo, Gerard, Joan, Jan. Past-tense \
narration. Lowell street names. Memory as primary subject.

TH4 MEXICO/PLACE/FELLAHEEN: Spanish vocabulary: ojo, palabra, Orizaba. \
Aztec mythological references: Huehueteotl, Quetzalcoatl, Popocatapetl. \
Fellaheen culture. Rooftop setting. Mock-Spanish coinages. Mexico City \
street life.

TH5 MORTALITY/BODY/SUFFERING: Morphine, physical illness, phlebitis, death \
as lived experience, bodily decay, addiction. Garver sequences. Dark \
imagery without Buddhist framing. Down-and-out bodily register.

TH6 META-POETRY/POETICS/SPONTANEITY: The act of composition is the subject. \
Parenthetical stage directions: (musician stops), (ripping of paper), \
(BLANK, the singer sings nothing). Words: chorus, canto, poem, blow, sing. \
A mundane action used as compositional analogy. Rest choruses.

Respond ONLY with valid JSON in this exact format:
{"tone": "T[number]", "theme": "TH[number]"}
No explanation. No commentary. No markdown backticks. Only the JSON object."""

print(f"System prompt defined.")
print(f"Character count: {len(SYSTEM_PROMPT)}")
print(f"Approximate token count: ~{len(SYSTEM_PROMPT)//4} tokens")

---
## Helper Functions

Two functions following the Week 10 `ask_llm()` pattern:

- **`ask_llm()`** — sends system prompt + user turn to Mistral via
  Ollama's Python client, returns the response string. Identical
  structure to Week 10, with a `system` parameter added.

- **`parse_annotation()`** — extracts tone and theme codes from the
  model's JSON response. Handles the most common failure mode: the
  model wrapping output in markdown backticks despite being told
  not to (observed in the connectivity check above).

In [ ]:
# ── Helper functions — following Week 10 pattern ──────────────────────

def ask_llm(prompt: str,
            system: str = None,
            model: str = TEXT_MODEL,
            temperature: float = TEMPERATURE) -> str:
    """
    Send a prompt to a local Ollama model and return the response string.

    Parameters
    ----------
    prompt : str
        The user message — one chorus with its ID number.
    system : str, optional
        System prompt carrying the coding scheme.
    model : str
        Ollama model name. Defaults to TEXT_MODEL ('mistral').
    temperature : float
        0.0 for deterministic annotation output.

    Returns
    -------
    str
        The model's response text, or '[ERROR] ...' on failure.
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    try:
        response = ollama.chat(
            model=model,
            messages=messages,
            options={"temperature": temperature},
        )
        return response["message"]["content"].strip()
    except Exception as e:
        return f"[ERROR] {e}"


def parse_annotation(response: str) -> dict:
    """
    Parse the LLM response into a dict with tone and theme codes.

    Handles markdown backtick wrapping, which Mistral occasionally
    produces despite being instructed not to.

    Returns {'tone': None, 'theme': None} if parsing fails — these
    are logged as annotation_success=False in the loop and never
    imputed or guessed.
    """
    result = {"tone": None, "theme": None}

    # Strip markdown backticks if present
    cleaned = response.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```[a-z]*\n?", "", cleaned)
        cleaned = re.sub(r"\n?```$", "", cleaned)
    cleaned = cleaned.strip()

    try:
        parsed = json.loads(cleaned)
        # Validate codes are in expected range before accepting
        tone  = parsed.get("tone", "")
        theme = parsed.get("theme", "")
        if tone  in {"T1", "T2", "T3", "T4", "T5"}:
            result["tone"] = tone
        if theme in {"TH1", "TH2", "TH3", "TH4", "TH5", "TH6"}:
            result["theme"] = theme
    except json.JSONDecodeError:
        pass

    return result


print("ask_llm() and parse_annotation() defined.")

In [ ]:
# ── Single-chorus test ────────────────────────────────────────────────
# Run before the full loop to verify prompt, model, and parse function
# all work together. Using Chorus 75 — a thematically significant
# chorus Jones discusses in Ch. 5, pp. 89-92.
# ─────────────────────────────────────────────────────────────────────

test_row = df[df["chorus_id"] == "75"].iloc[0]
test_prompt = f"Chorus {test_row['chorus_id']}:\n\n{test_row['text']}"

print("=== USER TURN SENT TO MODEL ===")
print(test_prompt)
print()

raw_response = ask_llm(test_prompt, system=SYSTEM_PROMPT)

print("=== RAW MODEL RESPONSE ===")
print(repr(raw_response))
print()

parsed = parse_annotation(raw_response)
print("=== PARSED RESULT ===")
print(f"  Tone:  {parsed['tone']}")
print(f"  Theme: {parsed['theme']}")
print()

# Interpret the result
tone_ok  = parsed["tone"]  is not None
theme_ok = parsed["theme"] is not None

if tone_ok and theme_ok:
    print("✓ Both labels parsed successfully.")
    print()
    print("Interpretation check (consult Jones Ch. 5, pp. 89-92):")
    print(f"  Model assigned: tone={parsed['tone']}, theme={parsed['theme']}")
    print("  Jones reads this as the singer's first epiphany that 'cantos")
    print("  oughta sing' — expect T2 (ECSTATIC) and TH2 (JAZZ/MUSIC)")
    print("  or TH6 (META-POETRY). Any of these is defensible.")
    print()
    print("✓ Ready to run the full annotation loop.")
else:
    print("⚠ Parsing failed — check the raw response above.")
    print("  Common fixes:")
    print("  1. Tighten the output constraint in SYSTEM_PROMPT")
    print("  2. Check that ollama serve is running in Terminal")
    print("  3. Try ollama pull mistral to ensure latest version")

In [ ]:
# ── 20-CHORUS TEST RUN ────────────────────────────────────────────────
# Run on the first 20 choruses only to check timing and output quality
# before committing to the full 244-chorus loop.
# Remove the [:20] slice when ready to run the full dataset.
# ─────────────────────────────────────────────────────────────────────

CHECKPOINT_PATH = Path("../outputs/annotations_checkpoint.csv")
CHECKPOINT_EVERY = 5  # save every 5 choruses during test run

# Work on first 20 choruses only
test_df = df[:20].copy()

# Add annotation columns to test_df if not present
for col in ["model_tone", "model_theme", "model_raw", "annotation_success"]:
    if col not in test_df.columns:
        test_df[col] = None

print(f"Running on {len(test_df)} choruses.")
print(f"Checkpoint saves to: {CHECKPOINT_PATH}")
print()

import time
start_time = time.time()

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Annotating"):

    # Skip if already annotated
    if pd.notna(test_df.at[idx, "model_tone"]):
        print(f"  Chorus {row['chorus_id']} already annotated — skipping.")
        continue

    # Build the user turn
    user_prompt = f"Chorus {row['chorus_id']}:\n\n{row['text']}"

    # First attempt
    raw = ask_llm(user_prompt, system=SYSTEM_PROMPT)
    result = parse_annotation(raw)

    # Re-prompt once if parsing failed
    if result["tone"] is None:
        print(f"  ⚠ Chorus {row['chorus_id']}: first attempt failed — re-prompting.")
        retry_prompt = (
            user_prompt +
            '\n\nYour previous response was not valid JSON. '
            'Respond ONLY with this exact format and nothing else: '
            '{"tone": "T[number]", "theme": "TH[number]"}'
        )
        raw = ask_llm(retry_prompt, system=SYSTEM_PROMPT)
        result = parse_annotation(raw)

    # Store results
    test_df.at[idx, "model_tone"]         = result["tone"]
    test_df.at[idx, "model_theme"]        = result["theme"]
    test_df.at[idx, "model_raw"]          = raw
    test_df.at[idx, "annotation_success"] = result["tone"] is not None

    # Checkpoint save every 5 choruses
    if (idx + 1) % CHECKPOINT_EVERY == 0:
        test_df.to_csv(CHECKPOINT_PATH, index=False)
        print(f"  Checkpoint saved at chorus {row['chorus_id']}")

# Final save
test_df.to_csv(CHECKPOINT_PATH, index=False)

# Timing report
elapsed = time.time() - start_time
per_chorus = elapsed / len(test_df)
estimated_full = per_chorus * 244

print()
print("=== TEST RUN COMPLETE ===")
print(f"  Choruses processed:  {len(test_df)}")
print(f"  Time taken:          {elapsed:.0f} seconds ({elapsed/60:.1f} minutes)")
print(f"  Time per chorus:     {per_chorus:.1f} seconds")
print(f"  Estimated full run:  {estimated_full:.0f} seconds ({estimated_full/60:.1f} minutes)")
print()
print(f"  Successful:  {test_df['annotation_success'].sum()}")
print(f"  Failed:      {(test_df['annotation_success'] == False).sum()}")
print()
print("Results preview:")
print(test_df[["chorus_id", "model_tone", "model_theme", "annotation_success"]].to_string())

In [ ]:
# ── SENSITIVITY CHECK ─────────────────────────────────────────────────
# Run on the first 20 choruses using two prompt variants.
# Variant A (baseline): uses 'egolessness' in T1, 'bop' in TH2.
# Variant B (alternate): replaces 'egolessness' with 'no-self'
#                        and 'bop' with 'jazz improvisation'.
# Consistency rate = proportion of choruses receiving identical labels
# across both variants. A rate below 85% is grounds for caution.
# Following Törnberg (2024) and Baumann et al. (2025) on prompt
# sensitivity as a required validity check.
# ─────────────────────────────────────────────────────────────────────

import time

# Variant B prompt — two controlled word substitutions only
SYSTEM_PROMPT_B = SYSTEM_PROMPT.replace(
    "egolessness", "no-self"
).replace(
    "T2 ECSTATIC/IMPROVISATORY: The chorus enacts jazz-inflected immediacy. Language \\
accelerates into sound-play. Dense oo-clusters, percussive coinages, \\
onomatopoeia, scat riffs. First-person and kinetic. A music reference or \\
musician name is present. Outward and fast-moving.",
    "T2 ECSTATIC/IMPROVISATORY: The chorus enacts jazz improvisation immediacy. Language \\
accelerates into sound-play. Dense oo-clusters, percussive coinages, \\
onomatopoeia, scat riffs. First-person and kinetic. A music reference or \\
musician name is present. Outward and fast-moving."
).replace(
    "bop, blues, jazz, horn",
    "jazz improvisation, blues, jazz, horn"
)

# Verify substitutions were made
assert "no-self" in SYSTEM_PROMPT_B, "no-self substitution failed"
print("Variant B system prompt constructed.")
print(f"Variant A length: {len(SYSTEM_PROMPT)} chars")
print(f"Variant B length: {len(SYSTEM_PROMPT_B)} chars")
print()

# Run on first 20 choruses
sense_df = df[:20].copy()[["chorus_id", "text"]]
sense_df["tone_A"]  = None
sense_df["theme_A"] = None
sense_df["tone_B"]  = None
sense_df["theme_B"] = None

print(f"Running sensitivity check on {len(sense_df)} choruses...")
print("(Each chorus queried twice — once per variant)")
print()

start = time.time()

for idx, row in tqdm(sense_df.iterrows(), total=len(sense_df), desc="Sensitivity"):
    user_prompt = f"Chorus {row['chorus_id']}:\n\n{row['text']}"

    # Variant A
    raw_a = ask_llm(user_prompt, system=SYSTEM_PROMPT)
    res_a = parse_annotation(raw_a)
    sense_df.at[idx, "tone_A"]  = res_a["tone"]
    sense_df.at[idx, "theme_A"] = res_a["theme"]

    # Variant B
    raw_b = ask_llm(user_prompt, system=SYSTEM_PROMPT_B)
    res_b = parse_annotation(raw_b)
    sense_df.at[idx, "tone_B"]  = res_b["tone"]
    sense_df.at[idx, "theme_B"] = res_b["theme"]

elapsed = time.time() - start

# Compute consistency rates
tone_match  = (sense_df["tone_A"]  == sense_df["tone_B"]).sum()
theme_match = (sense_df["theme_A"] == sense_df["theme_B"]).sum()
n = len(sense_df)

tone_rate  = tone_match  / n * 100
theme_rate = theme_match / n * 100

print()
print("=== SENSITIVITY CHECK RESULTS ===")
print(f"  Tone  consistency: {tone_match}/{n}  ({tone_rate:.1f}%)")
print(f"  Theme consistency: {theme_match}/{n}  ({theme_rate:.1f}%)")
print(f"  Time: {elapsed:.0f}s")
print()

if tone_rate >= 85 and theme_rate >= 85:
    print("✓ Consistency rates above 85% threshold.")
    print("  Prompt is stable across both variants. Proceed to full loop.")
else:
    print("⚠  Consistency rate below 85% on one or more dimensions.")
    print("  Review divergent choruses below before running the full loop.")
print()

# Show divergences
divergent = sense_df[
    (sense_df["tone_A"] != sense_df["tone_B"]) |
    (sense_df["theme_A"] != sense_df["theme_B"])
][["chorus_id", "tone_A", "tone_B", "theme_A", "theme_B"]]

if len(divergent) > 0:
    print(f"Divergent choruses ({len(divergent)}):")
    print(divergent.to_string(index=False))
else:
    print("No divergences — both variants produced identical output on all 20 choruses.")


In [ ]:
# ── SENSITIVITY CHECK ─────────────────────────────────────────────────
# Run on the first 20 choruses using two prompt variants.
# Variant A (baseline): uses 'egolessness' in T1, 'bop' in TH2.
# Variant B (alternate): replaces 'egolessness' with 'no-self'
#                        and 'bop' with 'jazz improvisation'.
# Consistency rate = proportion of choruses receiving identical labels
# across both variants. A rate below 85% is grounds for caution.
# Following Törnberg (2024) and Baumann et al. (2025) on prompt
# sensitivity as a required validity check.
# ─────────────────────────────────────────────────────────────────────

import time

# Variant B prompt — two controlled word substitutions only
SYSTEM_PROMPT_B = SYSTEM_PROMPT.replace(
    "egolessness", "no-self"
).replace(
    "T2 ECSTATIC/IMPROVISATORY: The chorus enacts jazz-inflected immediacy. Language \\
accelerates into sound-play. Dense oo-clusters, percussive coinages, \\
onomatopoeia, scat riffs. First-person and kinetic. A music reference or \\
musician name is present. Outward and fast-moving.",
    "T2 ECSTATIC/IMPROVISATORY: The chorus enacts jazz improvisation immediacy. Language \\
accelerates into sound-play. Dense oo-clusters, percussive coinages, \\
onomatopoeia, scat riffs. First-person and kinetic. A music reference or \\
musician name is present. Outward and fast-moving."
).replace(
    "bop, blues, jazz, horn",
    "jazz improvisation, blues, jazz, horn"
)

# Verify substitutions were made
assert "no-self" in SYSTEM_PROMPT_B, "no-self substitution failed"
print("Variant B system prompt constructed.")
print(f"Variant A length: {len(SYSTEM_PROMPT)} chars")
print(f"Variant B length: {len(SYSTEM_PROMPT_B)} chars")
print()

# Run on first 20 choruses
sense_df = df[:20].copy()[["chorus_id", "text"]]
sense_df["tone_A"]  = None
sense_df["theme_A"] = None
sense_df["tone_B"]  = None
sense_df["theme_B"] = None

print(f"Running sensitivity check on {len(sense_df)} choruses...")
print("(Each chorus queried twice — once per variant)")
print()

start = time.time()

for idx, row in tqdm(sense_df.iterrows(), total=len(sense_df), desc="Sensitivity"):
    user_prompt = f"Chorus {row['chorus_id']}:\n\n{row['text']}"

    # Variant A
    raw_a = ask_llm(user_prompt, system=SYSTEM_PROMPT)
    res_a = parse_annotation(raw_a)
    sense_df.at[idx, "tone_A"]  = res_a["tone"]
    sense_df.at[idx, "theme_A"] = res_a["theme"]

    # Variant B
    raw_b = ask_llm(user_prompt, system=SYSTEM_PROMPT_B)
    res_b = parse_annotation(raw_b)
    sense_df.at[idx, "tone_B"]  = res_b["tone"]
    sense_df.at[idx, "theme_B"] = res_b["theme"]

elapsed = time.time() - start

# Compute consistency rates
tone_match  = (sense_df["tone_A"]  == sense_df["tone_B"]).sum()
theme_match = (sense_df["theme_A"] == sense_df["theme_B"]).sum()
n = len(sense_df)

tone_rate  = tone_match  / n * 100
theme_rate = theme_match / n * 100

print()
print("=== SENSITIVITY CHECK RESULTS ===")
print(f"  Tone  consistency: {tone_match}/{n}  ({tone_rate:.1f}%)")
print(f"  Theme consistency: {theme_match}/{n}  ({theme_rate:.1f}%)")
print(f"  Time: {elapsed:.0f}s")
print()

if tone_rate >= 85 and theme_rate >= 85:
    print("✓ Consistency rates above 85% threshold.")
    print("  Prompt is stable across both variants. Proceed to full loop.")
else:
    print("⚠  Consistency rate below 85% on one or more dimensions.")
    print("  Review divergent choruses below before running the full loop.")
print()

# Show divergences
divergent = sense_df[
    (sense_df["tone_A"] != sense_df["tone_B"]) |
    (sense_df["theme_A"] != sense_df["theme_B"])
][["chorus_id", "tone_A", "tone_B", "theme_A", "theme_B"]]

if len(divergent) > 0:
    print(f"Divergent choruses ({len(divergent)}):")
    print(divergent.to_string(index=False))
else:
    print("No divergences — both variants produced identical output on all 20 choruses.")


In [ ]:
# ── CHECKPOINT RESUME ─────────────────────────────────────────────────
# Loads the checkpoint CSV if it exists and merges already-completed
# annotations back into df, so the full loop skips them.
# Safe to run even if no checkpoint exists — just starts fresh.
# ─────────────────────────────────────────────────────────────────────

CHECKPOINT_PATH = Path("../outputs/annotations_checkpoint.csv")

if CHECKPOINT_PATH.exists():
    ckpt = pd.read_csv(CHECKPOINT_PATH, dtype={"chorus_id": str})
    completed = ckpt[ckpt["annotation_success"] == True]["chorus_id"].tolist()

    # Merge completed annotations back into df
    annotated_cols = ["chorus_id", "model_tone", "model_theme",
                      "model_raw", "annotation_success"]
    ckpt_annotated = ckpt[annotated_cols]

    # Update df with checkpoint values
    df = df.set_index("chorus_id")
    for col in ["model_tone", "model_theme", "model_raw", "annotation_success"]:
        if col not in df.columns:
            df[col] = None
    ckpt_indexed = ckpt_annotated.set_index("chorus_id")
    df.update(ckpt_indexed)
    df = df.reset_index()

    print(f"Checkpoint loaded from: {CHECKPOINT_PATH}")
    print(f"  Choruses already completed: {len(completed)} / {len(df)}")
    print(f"  Remaining: {len(df) - len(completed)}")
else:
    # No checkpoint — ensure annotation columns exist
    for col in ["model_tone", "model_theme", "model_raw", "annotation_success"]:
        if col not in df.columns:
            df[col] = None
    print("No checkpoint found. Starting fresh from Chorus 1.")

print()
print("df ready for full annotation loop.")


In [ ]:
# ── FULL 244-CHORUS ANNOTATION LOOP ───────────────────────────────────
# Annotates all 244 choruses using Mistral 7B via Ollama.
# Skips choruses already annotated (checkpoint resume).
# Saves a checkpoint CSV every 25 choruses.
# Failed annotations (after one retry) are recorded as
# annotation_success=False and never imputed.
# ─────────────────────────────────────────────────────────────────────

CHECKPOINT_PATH  = Path("../outputs/annotations_checkpoint.csv")
FINAL_OUTPUT     = Path("../outputs/annotations_labels_only.csv")
CHECKPOINT_EVERY = 25

import time
start_time = time.time()
n_processed = 0
n_skipped   = 0
n_failed    = 0

print(f"Starting full annotation loop.")
print(f"Model:      {TEXT_MODEL}")
print(f"Temperature: {TEMPERATURE}")
print(f"Checkpoint: every {CHECKPOINT_EVERY} choruses → {CHECKPOINT_PATH}")
print(f"Final save: {FINAL_OUTPUT}")
print()

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Annotating MCB"):

    # Skip if already successfully annotated
    if df.at[idx, "annotation_success"] is True or df.at[idx, "annotation_success"] == True:
        n_skipped += 1
        continue

    # Build user turn
    user_prompt = f"Chorus {row['chorus_id']}:\n\n{row['text']}"

    # First attempt
    raw = ask_llm(user_prompt, system=SYSTEM_PROMPT)
    result = parse_annotation(raw)

    # One retry if parsing failed
    if result["tone"] is None:
        retry_prompt = (
            user_prompt
            + '\n\nYour previous response was not valid JSON. '
            'Respond ONLY with this exact format and nothing else: '
            '{"tone": "T[number]", "theme": "TH[number]"}'
        )
        raw = ask_llm(retry_prompt, system=SYSTEM_PROMPT)
        result = parse_annotation(raw)

    # Record outcome
    success = result["tone"] is not None
    df.at[idx, "model_tone"]         = result["tone"]
    df.at[idx, "model_theme"]        = result["theme"]
    df.at[idx, "model_raw"]          = raw
    df.at[idx, "annotation_success"] = success

    n_processed += 1
    if not success:
        n_failed += 1
        print(f"  ✗ Chorus {row['chorus_id']}: annotation failed after retry.")

    # Checkpoint every 25 choruses
    if n_processed % CHECKPOINT_EVERY == 0:
        df.to_csv(CHECKPOINT_PATH, index=False)
        elapsed = time.time() - start_time
        rate = elapsed / n_processed
        remaining = (len(df) - n_skipped - n_processed) * rate
        print(f"  ✓ Checkpoint at chorus {row['chorus_id']} "
              f"({n_processed} processed, ~{remaining/60:.0f} min remaining)")

# ── Final save ────────────────────────────────────────────────────────
df.to_csv(CHECKPOINT_PATH, index=False)
df.to_csv(FINAL_OUTPUT, index=False)

elapsed = time.time() - start_time

print()
print("=" * 50)
print("FULL ANNOTATION LOOP COMPLETE")
print("=" * 50)
print(f"  Total choruses:   {len(df)}")
print(f"  Skipped (cached): {n_skipped}")
print(f"  Newly annotated:  {n_processed}")
print(f"  Failed:           {n_failed}")
print(f"  Success rate:     {(n_processed - n_failed) / max(n_processed, 1) * 100:.1f}%")
print(f"  Total time:       {elapsed:.0f}s ({elapsed/60:.1f} min)")
print()
print(f"  Checkpoint: {CHECKPOINT_PATH}")
print(f"  Final CSV:  {FINAL_OUTPUT}")
print()

# Summary of label distribution
annotated = df[df["annotation_success"] == True]
print("Tone distribution:")
print(annotated["model_tone"].value_counts().sort_index().to_string())
print()
print("Theme distribution:")
print(annotated["model_theme"].value_counts().sort_index().to_string())
